# 01 — Dataset Inventory and Integrity Audit

**Objective:** Conduct an exhaustive file inventory, SHA-256 checksum audit, schema verification, data type validation, and duplicate analysis across the 5 annual raw CSV files (2011–2015).  
**Dataset Reference:** UCI Gas Turbine CO and NOx Emission Data Set (Heydar Kaya and Pınar Tüfekci).


In [1]:
import sys
import hashlib
import json
from pathlib import Path
import pandas as pd
import numpy as np

def find_root():
    p = Path.cwd()
    for parent in [p] + list(p.parents):
        if (parent / 'data' / 'raw' / 'gt_2011.csv').exists():
            return parent
    return p

ROOT = find_root()
sys.path.insert(0, str(ROOT))

from src.data.loader import EXPECTED_COLUMNS, EXPECTED_ROWS_BY_YEAR, TOTAL_EXPECTED_ROWS
from src.data.validation import validate_schema


## 1. SHA-256 Checksums and File Inventory
Computing cryptographic checksums and verifying uncompressed file sizes.


In [2]:
raw_dir = ROOT / "data" / "raw"
inventory = []

for year in range(2011, 2016):
    f_path = raw_dir / f"gt_{year}.csv"
    with open(f_path, "rb") as f:
        file_bytes = f.read()
        sha256 = hashlib.sha256(file_bytes).hexdigest()
    
    df_temp = pd.read_csv(f_path)
    inventory.append({
        "year": year,
        "filename": f_path.name,
        "file_size_bytes": f_path.stat().st_size,
        "sha256": sha256,
        "row_count": len(df_temp),
        "column_count": len(df_temp.columns),
        "expected_rows": EXPECTED_ROWS_BY_YEAR[year],
        "rows_match": len(df_temp) == EXPECTED_ROWS_BY_YEAR[year]
    })

inv_df = pd.DataFrame(inventory)
display_inv = inv_df[["year", "filename", "file_size_bytes", "row_count", "rows_match", "sha256"]]
print(display_inv.to_string(index=False))

# Export inventory CSV
out_inv = ROOT / "outputs" / "reports" / "dataset_inventory.csv"
inv_df.to_csv(out_inv, index=False)
print(f"Exported inventory to {out_inv.relative_to(ROOT)}")


 year    filename  file_size_bytes  row_count  rows_match                                                           sha256
 2011 gt_2011.csv           562921       7411        True d87ceef9aa59533cc7d924d10de241b1b06ecd11f9b26bab59191ea0f8a76b9a
 2012 gt_2012.csv           577617       7628        True be54b9d0e1a7de40c55d32fa489e75de892b000c066b5a09f09a19124ee29100
 2013 gt_2013.csv           540334       7152        True 13c437bb440ec2045bd12057e6654c41dd4107a661eac16ba2e878e897a08f9e
 2014 gt_2014.csv           541009       7158        True c2a03c92c9c3207aad0c6be7de8d9b5b4bfa4720ad0efb2c1f21b6cec4d3f3fa
 2015 gt_2015.csv           556928       7384        True 9b08f35fde0d4b138232a605db4093c2b8bf9d6757e6f1fbd9534ad616c13591
Exported inventory to outputs\reports\dataset_inventory.csv


## 2. Comprehensive Schema Audit
Validating exact column names, ordering, null counts, and floating-point data types.


In [3]:
schema_audit = {}
for year in range(2011, 2016):
    f_path = raw_dir / f"gt_{year}.csv"
    df_temp = pd.read_csv(f_path)
    val_res = validate_schema(df_temp, year=year)
    schema_audit[year] = val_res
    print(f"Year {year}: Valid={val_res['is_valid']}, Rows={val_res['row_count']}, Nulls={sum(val_res['null_counts'].values())}")

out_schema = ROOT / "outputs" / "reports" / "schema_audit.json"
with open(out_schema, "w", encoding="utf-8") as f:
    json.dump(schema_audit, f, indent=2)
print(f"Saved schema audit to {out_schema.relative_to(ROOT)}")


Year 2011: Valid=True, Rows=7411, Nulls=0
Year 2012: Valid=True, Rows=7628, Nulls=0
Year 2013: Valid=True, Rows=7152, Nulls=0
Year 2014: Valid=True, Rows=7158, Nulls=0
Year 2015: Valid=True, Rows=7384, Nulls=0
Saved schema audit to outputs\reports\schema_audit.json


## 3. Duplicate Record Audit
Auditing exact duplicate rows across all 11 sensor channels. In continuous turbine monitoring, identical sensor readings can occur during stationary base-load operations.


In [4]:
dup_records = []
for year in range(2011, 2016):
    df_temp = pd.read_csv(raw_dir / f"gt_{year}.csv")
    n_dups = int(df_temp.duplicated().sum())
    dup_records.append({
        "year": year,
        "total_rows": len(df_temp),
        "duplicate_rows": n_dups,
        "duplicate_pct": round(n_dups / len(df_temp) * 100, 3)
    })

dup_df = pd.DataFrame(dup_records)
print(dup_df.to_string(index=False))

out_dup = ROOT / "outputs" / "tables" / "duplicate_audit.csv"
out_dup.parent.mkdir(parents=True, exist_ok=True)
dup_df.to_csv(out_dup, index=False)
print(f"Saved duplicate audit to {out_dup.relative_to(ROOT)}")


 year  total_rows  duplicate_rows  duplicate_pct
 2011        7411               0          0.000
 2012        7628               0          0.000
 2013        7152               0          0.000
 2014        7158               7          0.098
 2015        7384               0          0.000
Saved duplicate audit to outputs\tables\duplicate_audit.csv


## 4. Full Data Quality Report Export
Aggregating statistical distribution summaries across all 36,733 observations.


In [5]:
from src.data.loader import load_raw_yearly_data
from src.data.quality_report import generate_data_quality_report

all_df = load_raw_yearly_data(raw_dir)
quality_report = generate_data_quality_report(
    all_df, 
    output_path=ROOT / "outputs" / "reports" / "data_quality_report.json"
)

print(f"Total dataset observations: {quality_report['dataset_dimensions']['total_observations']:,}")
print(f"Zero missing values confirmed across all 11 columns.")
print("Dataset inventory and integrity audit COMPLETE.")


Total dataset observations: 36,733
Zero missing values confirmed across all 11 columns.
Dataset inventory and integrity audit COMPLETE.
